# Geolocation using Nominatim
We'll import Nominatim from GeoPy and also a library that handles a large number of queries. The idea is to read the CSV of Lisbon, collect the coordinates, and add columns for these locations.

In [40]:
%config IPCompleter.greedy=True #autocomplete

!["Documentação GeoPy: Uso de Rate Limiter"](Snipaste_2025-07-07_08-09-03.png)

In [41]:
import geopy.geocoders
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd
import ssl
import certifi

Initializing the library with a localizer variable from which we'll apply functions, and reading the table, transforming it into a DataFrame.

In [42]:
ctx = ssl.create_default_context(cafile=certifi.where())
geopy.geocoders.options.default_ssl_context = ctx
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

localizer = Nominatim(scheme = 'http', user_agent="my_geolocation_bars", timeout=10)
geocode = RateLimiter(localizer.geocode, min_delay_seconds=1)

df = pd.read_csv('lisbon_places_normalized.csv')
df

,name,category,partial_address,ref,website
0,TRUMPS,nightclub/bar,"TRUMPS, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN
1,FINALMENTE CLUBE,nightclub/bar,"FINALMENTE CLUBE, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN
2,CONSTRUCTION,nightclub/bar,"CONSTRUCTION, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN
3,PRESTIGE,nightclub/bar,"PRESTIGE, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN
4,106,nightclub/bar,"106, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN
5,BAR CRU,nightclub/bar,"BAR CRU, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN
6,CONSTRUCTION BAR - BAIRRO ALTO,nightclub/bar,"CONSTRUCTION BAR – BAIRRO ALTO, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN
7,ENTRETANTO ROOFTOP,nightclub/bar,"ENTRETANTO ROOFTOP, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN
8,TR3S,nightclub/bar,"TR3S, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN
9,SHELTER BAR,nightclub/bar,"SHELTER BAR, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN


We'll create a list of locations to store the coordinates by iterating through each row of the DataFrame.

In [43]:
locations = []
for i, row in df.iterrows():
    partial_address = row['partial_address']
    location = geocode(partial_address)
    locations.append(location)
    if location:
        print(f"-> found!: {partial_address}")
    else:
        location = None
        print(f"-> NOT found.{partial_address}")

print("\ngeocoding finished.")
print(locations)

-> found!: TRUMPS, Lisboa - Portugal
-> found!: FINALMENTE CLUBE, Lisboa - Portugal
-> found!: CONSTRUCTION, Lisboa - Portugal
-> found!: PRESTIGE, Lisboa - Portugal
-> found!: 106, Lisboa - Portugal
-> found!: BAR CRU, Lisboa - Portugal
-> NOT found.CONSTRUCTION BAR – BAIRRO ALTO, Lisboa - Portugal
-> NOT found.ENTRETANTO ROOFTOP, Lisboa - Portugal
-> found!: TR3S, Lisboa - Portugal
-> found!: SHELTER BAR, Lisboa - Portugal
-> found!: SIDE BAR, Lisboa - Portugal
-> NOT found.MINIGOLF LISBON, Lisboa - Portugal
-> found!: XXL, Lisboa - Portugal
-> NOT found.DARK BY NUDE – ALBUFEIRA, Lisboa - Portugal
-> NOT found.THE FOREST – ALBUFEIRA, Lisboa - Portugal
-> found!: TROMBETA BATH, Lisboa - Portugal
-> NOT found.SAUNAPOLO 56, Lisboa - Portugal
-> NOT found.OLISSIPO BATH, Lisboa - Portugal
-> NOT found.3SAUNA, Lisboa - Portugal
-> found!: BISTRO EDELWEISS, Lisboa - Portugal
-> found!: CASA DA PRAIA, Lisboa - Portugal
-> found!: DR. BERNARD, Lisboa - Portugal
-> NOT found.GLITTER’S GALLERY,

We'll add latitude, longitude, point (latitude and longitude), and the establishment's name collected by Nominatim (to check if it matches what was searched) as columns to the DataFrame.

In [44]:
df['geolocated_address'] = [loc.address if loc else None for loc in locations]
df['latitude'] = [loc.latitude if loc else None for loc in locations]
df['longitude'] = [loc.longitude if loc else None for loc in locations]
df['point'] = [tuple(loc.point) if loc else None for loc in locations]
df = df.rename(columns = {'categorie':'category'})
df

,name,category,partial_address,ref,website,geolocated_address,latitude,longitude,point
0,TRUMPS,nightclub/bar,"TRUMPS, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN,"Trumps Lisbon, 104B, Rua da Imprensa Nacional,...",38.717415,-9.151900,"(38.7174149, -9.1519004, 0.0)"
1,FINALMENTE CLUBE,nightclub/bar,"FINALMENTE CLUBE, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN,"Finalmente Clube, 38, Rua da Palmeira, Mercês,...",38.714790,-9.149876,"(38.7147899, -9.1498756, 0.0)"
2,CONSTRUCTION,nightclub/bar,"CONSTRUCTION, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN,Construção do Túnel de Drenagem Monsanto-Santa...,38.735001,-9.167485,"(38.7350007, -9.1674845, 0.0)"
3,PRESTIGE,nightclub/bar,"PRESTIGE, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN,"Prestige, Rua das Acácias, Jardins da Parede, ...",38.695718,-9.365295,"(38.6957179, -9.3652949, 0.0)"
4,106,nightclub/bar,"106, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN,"106, Avenida da República, Alvalade, Lisboa, 1...",38.747736,-9.147759,"(38.7477356, -9.1477589, 0.0)"
5,BAR CRU,nightclub/bar,"BAR CRU, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN,"Bar Cru, 170, Rua de São Marçal, Mercês, Santo...",38.715988,-9.150327,"(38.7159879, -9.1503266, 0.0)"
6,CONSTRUCTION BAR - BAIRRO ALTO,nightclub/bar,"CONSTRUCTION BAR – BAIRRO ALTO, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN,None,NaN,NaN,None
7,ENTRETANTO ROOFTOP,nightclub/bar,"ENTRETANTO ROOFTOP, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN,None,NaN,NaN,None
8,TR3S,nightclub/bar,"TR3S, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN,"TR3S Lisboa, 2, Rua Ruben A. Leitão, Mercês, M...",38.714329,-9.149425,"(38.714329, -9.1494246, 0.0)"
9,SHELTER BAR,nightclub/bar,"SHELTER BAR, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,NaN,"Shelter Bar, Rua da Palmeira, Mercês, Misericó...",38.714848,-9.150521,"(38.7148479, -9.1505212, 0.0)"


In [45]:
df.to_csv('geolocated_places_uncomplete.csv', index=False, encoding='utf-8-sig')

## Manually completing missing addresses
With the .csv table ready, we'll manually add the few missing addresses

In [46]:
# table edited to add missing addresses 
df_completed = pd.read_csv('geolocated_places_uncomplete_edited.csv')
df_completed

,name,category,partial_address,ref,website,geolocated_address,latitude,longitude,point
0,TRUMPS,nightclub/bar,"TRUMPS, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"Trumps Lisbon, 104B, Rua da Imprensa Nacional,...",38.717415,-9.151900,"(38.7174149, -9.1519004, 0.0)"
1,FINALMENTE CLUBE,nightclub/bar,"FINALMENTE CLUBE, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"Finalmente Clube, 38, Rua da Palmeira, Mercês,...",38.714790,-9.149876,"(38.7147899, -9.1498756, 0.0)"
2,CONSTRUCTION,nightclub/bar,"Rua da Rosa 157, Lisboa, Portugal",https://www.lisbongaycircuit.com/#\t,NaN,NaN,NaN,NaN,NaN
3,PRESTIGE,nightclub/bar,"PRESTIGE, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"Prestige, Rua das Acácias, Jardins da Parede, ...",38.695718,-9.365295,"(38.6957179, -9.3652949, 0.0)"
4,106,nightclub/bar,"106, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"106, Avenida da República, Alvalade, Lisboa, 1...",38.747736,-9.147759,"(38.7477356, -9.1477589, 0.0)"
5,BAR CRU,nightclub/bar,"BAR CRU, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"Bar Cru, 170, Rua de São Marçal, Mercês, Santo...",38.715988,-9.150327,"(38.7159879, -9.1503266, 0.0)"
6,CONSTRUCTION BAR - BAIRRO ALTO,nightclub/bar,"CONSTRUCTION BAR – BAIRRO ALTO, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,NaN,NaN,NaN,NaN
7,ENTRETANTO ROOFTOP,nightclub/bar,"Rua Nova do Almada 114, Lisboa, Portugal",https://www.lisbongaycircuit.com/#\t,NaN,NaN,NaN,NaN,NaN
8,TR3S,nightclub/bar,"TR3S, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"TR3S Lisboa, 2, Rua Ruben A. Leitão, Mercês, M...",38.714329,-9.149425,"(38.714329, -9.1494246, 0.0)"
9,SHELTER BAR,nightclub/bar,"SHELTER BAR, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"Shelter Bar, Rua da Palmeira, Mercês, Misericó...",38.714848,-9.150521,"(38.7148479, -9.1505212, 0.0)"


In [47]:
locations = []
for i, row in df_completed.iterrows():
    partial_address = row['partial_address']
    location = geocode(partial_address)
    locations.append(location)
    if location:
        print(f"-> found!: {partial_address}")
    else:
        location = None
        print(f"-> NOT found.{partial_address}")

print("\ngeocoding finished.")

-> found!: TRUMPS, Lisboa - Portugal
-> found!: FINALMENTE CLUBE, Lisboa - Portugal
-> found!: Rua da Rosa 157, Lisboa, Portugal
-> found!: PRESTIGE, Lisboa - Portugal
-> found!: 106, Lisboa - Portugal
-> found!: BAR CRU, Lisboa - Portugal
-> NOT found.CONSTRUCTION BAR – BAIRRO ALTO, Lisboa - Portugal
-> found!: Rua Nova do Almada 114, Lisboa, Portugal
-> found!: TR3S, Lisboa - Portugal
-> found!: SHELTER BAR, Lisboa - Portugal
-> found!: SIDE BAR, Lisboa - Portugal
-> found!: XXL, Lisboa - Portugal
-> NOT found.DARK BY NUDE – ALBUFEIRA, Lisboa - Portugal
-> NOT found.THE FOREST – ALBUFEIRA, Lisboa - Portugal
-> found!: TROMBETA BATH, Lisboa - Portugal
-> found!:  Rua Luciano Cordeiro 56, Lisboa - Portugal
-> found!: Rua do Telhal 5, Lisboa - Portugal
-> found!: BISTRO EDELWEISS, Lisboa - Portugal
-> found!: CASA DA PRAIA, Lisboa - Portugal
-> found!: DR. BERNARD, Lisboa - Portugal
-> found!: Avenida Duque de Loulé 51B, Lisboa - Portugal
-> found!: Rua da Barroca, 92. BAIRRO ALTO, Lisb

In [48]:
df_completed['geolocated_address'] = [loc.address if loc else None for loc in locations]
df_completed['latitude'] = [loc.latitude if loc else None for loc in locations]
df_completed['longitude'] = [loc.longitude if loc else None for loc in locations]
df_completed['point'] = [tuple(loc.point) if loc else None for loc in locations]
df_completed = df_completed.rename(columns = {'categorie':'category'})
df_completed

,name,category,partial_address,ref,website,geolocated_address,latitude,longitude,point
0,TRUMPS,nightclub/bar,"TRUMPS, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"Trumps Lisbon, 104B, Rua da Imprensa Nacional,...",38.717415,-9.151900,"(38.7174149, -9.1519004, 0.0)"
1,FINALMENTE CLUBE,nightclub/bar,"FINALMENTE CLUBE, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"Finalmente Clube, 38, Rua da Palmeira, Mercês,...",38.714790,-9.149876,"(38.7147899, -9.1498756, 0.0)"
2,CONSTRUCTION,nightclub/bar,"Rua da Rosa 157, Lisboa, Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"Rua da Rosa, Bairro Alto, Misericórdia, Lisboa...",38.713402,-9.145462,"(38.7134023, -9.1454618, 0.0)"
3,PRESTIGE,nightclub/bar,"PRESTIGE, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"Prestige, Rua das Acácias, Jardins da Parede, ...",38.695718,-9.365295,"(38.6957179, -9.3652949, 0.0)"
4,106,nightclub/bar,"106, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"106, Avenida da República, Alvalade, Lisboa, 1...",38.747736,-9.147759,"(38.7477356, -9.1477589, 0.0)"
5,BAR CRU,nightclub/bar,"BAR CRU, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"Bar Cru, 170, Rua de São Marçal, Mercês, Santo...",38.715988,-9.150327,"(38.7159879, -9.1503266, 0.0)"
6,CONSTRUCTION BAR - BAIRRO ALTO,nightclub/bar,"CONSTRUCTION BAR – BAIRRO ALTO, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,None,NaN,NaN,None
7,ENTRETANTO ROOFTOP,nightclub/bar,"Rua Nova do Almada 114, Lisboa, Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"Hotel do Chiado, 114, Rua Nova do Almada, Chia...",38.710720,-9.139489,"(38.7107197, -9.1394885, 0.0)"
8,TR3S,nightclub/bar,"TR3S, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"TR3S Lisboa, 2, Rua Ruben A. Leitão, Mercês, M...",38.714329,-9.149425,"(38.714329, -9.1494246, 0.0)"
9,SHELTER BAR,nightclub/bar,"SHELTER BAR, Lisboa - Portugal",https://www.lisbongaycircuit.com/#\t,NaN,"Shelter Bar, Rua da Palmeira, Mercês, Misericó...",38.714848,-9.150521,"(38.7148479, -9.1505212, 0.0)"


In [49]:
df_completed.to_csv('geolocated_places_complete.csv', index=False, encoding='utf-8-sig' )

# Plotando bares em um mapa usando Folium

In [50]:
import folium

categories_colors = {
    'nightclub/bar': 'pink',       
    'sauna': 'red',              
    'restaurant': 'orange',        
    'association': 'purple',      
    'accommodation': 'green',    
    'beach': 'ciano',        
    }

default_color = 'gray'
center = [38.7223, -9.1393]
lisbon_map = folium.Map(location=center, zoom_start=14)

for i, row in df.iterrows():
    if pd.notna(row['latitude']) and pd.notna(row['longitude']):
        coordinates = [row['latitude'], row['longitude']]
        name = row['name']
        address = row['geolocated_address']
        category = row['category']
        marker_color = categories_colors.get(category, default_color)
        html_popup = f"""
        <h4>{name}</h4>
        <p>
        <b>Address: </b> {address}<br>
        </p>
        <p>
        <b>Category: </b> {category}<br>
        </p>
        """
        popup = folium.Popup(html_popup, max_width=250)
        folium.Marker(location=coordinates, popup=popup, tooltip=name, icon=folium.Icon(color=marker_color, icon='info-sign') ).add_to(lisbon_map)

lisbon_map.save('lisbon_places_map.html')
lisbon_map